In [11]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx
import imageio
import os
from simpledbf import Dbf5
import openpyxl
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.cluster import KMeans

PyTables is not installed. No support for HDF output.
SQLalchemy is not installed. No support for SQL output.


In [40]:
agebs_gdf = gpd.read_file("26/ageb_urb.shp").to_crs("4326").cx[-111.25:-110.75, 28.9:29.3].reset_index()
agebs_gdf.head()

,index,CVEGEO,POB1,POB2,POB2_R,POB3,POB3_R,POB4,POB4_R,POB5,...,POB78,POB78_R,POB79,POB79_R,POB80,POB80_R,POB81,POB81_R,OID,geometry
0,1016,2603000010023,0,-6,-6.0,-6,-6.0,-6,-6.0,-6,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,1017,"POLYGON ((-110.99153 29.06045, -110.99091 29.0..."
1,1017,2603000010076,6,-6,-6.0,-6,-6.0,-6,-6.0,-6,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,1018,"POLYGON ((-111.03254 29.1459, -111.03275 29.14..."
2,1018,2603000010080,0,0,-8.0,0,-8.0,0,-8.0,0,...,0,-8.0,0,-8.0,0,-8.0,0,-8.0,1019,"POLYGON ((-111.02556 29.16581, -111.02558 29.1..."
3,1019,2603000010095,218,22,10.1,32,14.7,13,6.0,19,...,53,49.1,0,0.0,0,0.0,0,0.0,1020,"POLYGON ((-111.00192 29.16435, -111.00197 29.1..."
4,1020,2603000010108,0,0,-8.0,0,-8.0,0,-8.0,0,...,0,-8.0,0,-8.0,0,-8.0,0,-8.0,1021,"POLYGON ((-110.98598 29.16559, -110.9871 29.16..."


In [43]:
agebs_dbf = Dbf5("./26/ageb_urb.dbf")
agebs_df = agebs_dbf.to_dataframe()
agebs_df.head()

,CVEGEO,POB1,POB2,POB2_R,POB3,POB3_R,POB4,POB4_R,POB5,POB5_R,...,POB77_R,POB78,POB78_R,POB79,POB79_R,POB80,POB80_R,POB81,POB81_R,OID
0,2600100010055,854,28,3.3,52,6.1,42,4.9,107,12.5,...,65.5,243,52.4,47,10.1,36,7.8,23,5.0,1
1,260010001006A,747,33,4.4,57,7.6,33,4.4,76,10.2,...,71.6,256,64.3,87,21.9,63,15.8,46,11.6,2
2,2600100010074,50,-6,-6.0,6,12.0,5,10.0,11,22.0,...,40.0,11,36.7,-6,-6.0,-6,-6.0,0,0.0,3
3,2600100010110,63,3,4.8,5,7.9,8,12.7,11,17.5,...,48.1,11,40.7,0,0.0,0,0.0,0,0.0,4
4,2600100010125,0,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,5


In [48]:
agebs_df["Clave de AGEB"] = agebs_df["CVEGEO"].str[9:]
agebs_df

,CVEGEO,POB1,POB2,POB2_R,POB3,POB3_R,POB4,POB4_R,POB5,POB5_R,...,POB78,POB78_R,POB79,POB79_R,POB80,POB80_R,POB81,POB81_R,OID,Clave de AGEB
0,2600100010055,854,28,3.3,52,6.1,42,4.9,107,12.5,...,243,52.4,47,10.1,36,7.8,23,5.0,1,0055
1,260010001006A,747,33,4.4,57,7.6,33,4.4,76,10.2,...,256,64.3,87,21.9,63,15.8,46,11.6,2,006A
2,2600100010074,50,-6,-6.0,6,12.0,5,10.0,11,22.0,...,11,36.7,-6,-6.0,-6,-6.0,0,0.0,3,0074
3,2600100010110,63,3,4.8,5,7.9,8,12.7,11,17.5,...,11,40.7,0,0.0,0,0.0,0,0.0,4,0110
4,2600100010125,0,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,5,0125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2725,2607200170269,63,6,9.5,8,12.7,4,6.3,8,12.7,...,13,41.9,0,0.0,0,0.0,0,0.0,2726,0269
2726,2607200170273,7,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2727,0273
2727,2607200170288,7,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2728,0288
2728,2607200170292,9,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2729,0292


In [45]:
population_df = pd.read_excel("./26/tablas/RESAGEB2010 - 26 Sonora.xlsx", header = 3)
population_df.columns = population_df.iloc[0]
population_df

,Clave de entidad federativa,Nombre de la entidad,Clave de municipio o delegación,Nombre del municipio o delegación,Clave de localidad,Nombre de la localidad,Clave de AGEB,Clave de la manzana,Población total
0,Clave de entidad federativa,Nombre de la entidad,Clave de municipio o delegación,Nombre del municipio o delegación,Clave de localidad,Nombre de la localidad,Clave de AGEB,Clave de la manzana,Población total
1,26,Sonora,030,Hermosillo,0000,TOTAL DEL MUNICIPIO,0000,000,784342
2,26,Sonora,030,Hermosillo,0001,Total de la localidad urbana,0000,000,715061
3,26,Sonora,030,Hermosillo,0001,Total AGEB urbana,0023,000,0
4,26,Sonora,030,Hermosillo,0001,Hermosillo,0023,001,0
...,...,...,...,...,...,...,...,...,...
14885,26,Sonora,030,Hermosillo,0535,San Pedro o el Saucito (San Pedro el Saucito),8851,002,21
14886,26,Sonora,030,Hermosillo,0535,San Pedro o el Saucito (San Pedro el Saucito),8851,003,8
14887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
population_df = population_df[population_df["Nombre de la localidad"] == "Total AGEB urbana"]
population_df = population_df[population_df["Nombre del municipio o delegación"] == "Hermosillo"]
population_df = population_df[["Clave de AGEB", "Población total"]]
population_df.sort_values("Población total", ascending = False)

,Clave de AGEB,Población total
1520,2246,13525
3201,2710,7334
5920,4863,6865
1855,2335,6844
10231,622A,6584
...,...,...
12838,8118,0
12914,8245,0
12836,8103,0
12742,8048,0


In [39]:
population_df["Población total"].sum()

754918

In [52]:
pop_df = agebs_df.join(population_df.set_index("Clave de AGEB"), on = "Clave de AGEB")
pop_df

,CVEGEO,POB1,POB2,POB2_R,POB3,POB3_R,POB4,POB4_R,POB5,POB5_R,...,POB78_R,POB79,POB79_R,POB80,POB80_R,POB81,POB81_R,OID,Clave de AGEB,Población total
0,2600100010055,854,28,3.3,52,6.1,42,4.9,107,12.5,...,52.4,47,10.1,36,7.8,23,5.0,1,0055,NaN
1,260010001006A,747,33,4.4,57,7.6,33,4.4,76,10.2,...,64.3,87,21.9,63,15.8,46,11.6,2,006A,NaN
2,2600100010074,50,-6,-6.0,6,12.0,5,10.0,11,22.0,...,36.7,-6,-6.0,-6,-6.0,0,0.0,3,0074,NaN
3,2600100010110,63,3,4.8,5,7.9,8,12.7,11,17.5,...,40.7,0,0.0,0,0.0,0,0.0,4,0110,NaN
4,2600100010125,0,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,5,0125,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2725,2607200170269,63,6,9.5,8,12.7,4,6.3,8,12.7,...,41.9,0,0.0,0,0.0,0,0.0,2726,0269,330
2726,2607200170273,7,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2727,0273,0
2727,2607200170288,7,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2728,0288,4
2728,2607200170292,9,-6,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,...,-6.0,-6,-6.0,-6,-6.0,-6,-6.0,2729,0292,2


In [53]:
pop_df.isna().sum()

CVEGEO                0
POB1                  0
POB2                  0
POB2_R                0
POB3                  0
                   ... 
POB81                 0
POB81_R               0
OID                   0
Clave de AGEB         0
Población total    2104
Length: 160, dtype: int64